# Load in Data

In [1]:
import pandas as pd

# Load features
features_df = pd.read_csv('../models/features.csv', index_col=0)

# Load selected feature names
with open('selected_features_top50.txt', 'r') as f:
    selected_features = [line.strip() for line in f if line.strip()]

features_df = features_df[selected_features + ['DQ_TARGET']]
features_df = features_df[features_df["DQ_TARGET"].notnull()].copy()

print(f'Total consumers: {features_df.shape[0]}')
print(f'Selected features: {len(selected_features)}')
print(f'Delinquency rate: {features_df["DQ_TARGET"].mean():.2%}')

Total consumers: 10317
Selected features: 50
Delinquency rate: 8.86%


# Scoring Exclusion

In [2]:
import numpy as np
import pandas as pd

df_work = features_df.copy()
LABEL = "DQ_TARGET"

# Starting counts
start_total = len(df_work)
start_labeled = df_work[LABEL].notnull().sum()

# Track removals
removed_by_rule = {}

# Running total removed
running_removed = 0

print("Starting total:", start_total)
print("Starting labeled:", start_labeled)

Starting total: 10317
Starting labeled: 10317


### Minimum History Days

In [3]:
MIN_DAYS = 20

mask = (df_work["n_days__all"] < MIN_DAYS).fillna(True)

removed = int(mask.sum())
removed_by_rule["rule_1_min_days"] = removed

df_work = df_work.loc[~mask].copy()

running_removed += removed

print("Rule 1 removed:", removed)

Rule 1 removed: 982


### Minimum Recent Transactions

In [4]:
MIN_TX_30D = 5

mask = (df_work["n_tx__30d"] < MIN_TX_30D).fillna(True)

removed = int(mask.sum())
removed_by_rule["rule_2_min_tx30d"] = removed

df_work = df_work.loc[~mask].copy()

running_removed += removed

print("Rule 2 removed:", removed)


Rule 2 removed: 95


### Income Signal Present

In [5]:
MIN_INCOME_FREQ = 2/90 # 2 every 90 days

mask = (df_work["income__frequency__90d"] < MIN_INCOME_FREQ).fillna(True)

removed = int(mask.sum())
removed_by_rule["rule_3_income_presence"] = removed

df_work = df_work.loc[~mask].copy()

running_removed += removed

print("Rule 3 removed:", removed)

Rule 3 removed: 372


### Credit transaction info present

In [6]:
mask = (df_work["tx__max_credit__all"] <= 0).fillna(True)

removed = int(mask.sum())
removed_by_rule["rule_4_has_credit"] = removed

df_work = df_work.loc[~mask].copy()

running_removed += removed

print("Rule 4 removed:", removed)

Rule 4 removed: 0


In [7]:
final_total = len(df_work)
final_labeled = df_work[LABEL].notnull().sum()

print("----- SUMMARY -----")
print("Started with:", start_total)
print("Total removed:", running_removed)
print("Remaining:", final_total)
print()

print("Removed by rule:")
for k, v in removed_by_rule.items():
    print(k, ":", v)

print()
print("Labeled remaining:", final_labeled)
print(f'Percent Removed: {round(running_removed/start_total,5) * 100}%')

----- SUMMARY -----
Started with: 10317
Total removed: 1449
Remaining: 8868

Removed by rule:
rule_1_min_days : 982
rule_2_min_tx30d : 95
rule_3_income_presence : 372
rule_4_has_credit : 0

Labeled remaining: 8868
Percent Removed: 14.045%


# Data Splitting

In [8]:
from sklearn.model_selection import train_test_split
import numpy as np

LABEL = "DQ_TARGET"

df_clean = df_work[df_work[LABEL].notna()].copy()
X = df_clean.drop(columns=[LABEL]).fillna(0).values
y = df_clean[LABEL].astype(int).values

# 60% train, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# 20% val, 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train pos rate:", y_train.mean())
print("Val pos rate:", y_val.mean())
print("Test pos rate:", y_test.mean())

Train pos rate: 0.08609022556390977
Val pos rate: 0.08624577226606539
Test pos rate: 0.08624577226606539


In [9]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE: ", np.bincount(y_train_sm))

Before SMOTE: [4862  458]
After SMOTE:  [4862 4862]


In [10]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def metrics_df(model, X, y, split_name, model_name, train_time_sec):
    y_pred = model.predict(X)

    # ROC AUC requires probabilities
    y_prob = model.predict_proba(X)[:, 1]
    roc = roc_auc_score(y, y_prob)

    return pd.DataFrame({
        "model": [model_name],
        "split": [split_name],
        "accuracy": [accuracy_score(y, y_pred)],
        "roc_auc": [roc],
        "f1": [f1_score(y, y_pred, zero_division=0)],
        "train_time_sec": [train_time_sec],
    })

# Model Performance

# Decision Tree

In [11]:
from sklearn.tree import DecisionTreeClassifier
import time

t0 = time.time()
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=25,
    random_state=42
)
dt.fit(X_train_sm, y_train_sm)
dt_train_time = time.time() - t0

dt_results = pd.concat([
    metrics_df(dt, X_train, y_train, "train", "DecisionTree", dt_train_time),
    metrics_df(dt, X_val, y_val, "validation", "DecisionTree", dt_train_time),
    metrics_df(dt, X_test, y_test, "test", "DecisionTree", dt_train_time),
], ignore_index=True)

dt_results

,model,split,accuracy,roc_auc,f1,train_time_sec
0,DecisionTree,train,0.833459,0.766753,0.265340,0.288656
1,DecisionTree,validation,0.817926,0.685710,0.194514,0.288656
2,DecisionTree,test,0.822435,0.715338,0.214464,0.288656


# Random Forest

In [12]:
from sklearn.ensemble import RandomForestClassifier
import time

t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_sm, y_train_sm)
rf_train_time = time.time() - t0

rf_results = pd.concat([
    metrics_df(rf, X_train, y_train, "train", "RandomForest", rf_train_time),
    metrics_df(rf, X_val, y_val, "validation", "RandomForest", rf_train_time),
    metrics_df(rf, X_test, y_test, "test", "RandomForest", rf_train_time),
], ignore_index=True)

rf_results

,model,split,accuracy,roc_auc,f1,train_time_sec
0,RandomForest,train,0.947744,0.963967,0.707983,3.213886
1,RandomForest,validation,0.865840,0.762291,0.287425,3.213886
2,RandomForest,test,0.859076,0.773645,0.289773,3.213886


In [13]:
all_results = pd.concat([dt_results, rf_results], ignore_index=True)
all_results = all_results.drop(columns=["train_time_sec"], errors="ignore")
all_results

,model,split,accuracy,roc_auc,f1
0,DecisionTree,train,0.833459,0.766753,0.265340
1,DecisionTree,validation,0.817926,0.685710,0.194514
2,DecisionTree,test,0.822435,0.715338,0.214464
3,RandomForest,train,0.947744,0.963967,0.707983
4,RandomForest,validation,0.865840,0.762291,0.287425
5,RandomForest,test,0.859076,0.773645,0.289773


In [14]:
import pandas as pd
import numpy as np

# Decision Tree baseline (fill these in)
dt_baseline = pd.DataFrame({
    "model": ["DecisionTree", "DecisionTree", "DecisionTree"],
    "split": ["train", "validation", "test"],
    "accuracy": [0.752, 0.721, 0.711],      
    "roc_auc": [0.69, 0.621, 0.629],       
    "f1": [0.24, 0.165, 0.164],            
    "train_time_sec": [2.51, 2.51, 2.51] 
})

# Random Forest baseline (your values)
rf_baseline = pd.DataFrame({
    "model": ["RandomForest", "RandomForest", "RandomForest"],
    "split": ["train", "validation", "test"],
    "accuracy": [0.96, 0.87, 0.88],
    "roc_auc": [0.9850, 0.7925, 0.7663],
    "f1": [0.79, 0.29, 0.29],
    "train_time_sec": [2.86, 2.86, 2.86]
})

baseline_results = pd.concat([dt_baseline, rf_baseline], ignore_index=True)
baseline_results = baseline_results.drop(columns=["train_time_sec"], errors="ignore")
baseline_results

,model,split,accuracy,roc_auc,f1
0,DecisionTree,train,0.752,0.6900,0.240
1,DecisionTree,validation,0.721,0.6210,0.165
2,DecisionTree,test,0.711,0.6290,0.164
3,RandomForest,train,0.960,0.9850,0.790
4,RandomForest,validation,0.870,0.7925,0.290
5,RandomForest,test,0.880,0.7663,0.290
